# Multi-Agent Orchestrator — Demo Notebook

This notebook demonstrates the full pipeline of the LangGraph-based multi-agent orchestrator.  
It walks through:

1. Setup and environment configuration  
2. Graph structure visualization  
3. Three example queries covering different agent capabilities  

## Cell 1 — Setup: Imports & Environment

In [ ]:
import os
import sys
import json

# Load environment variables from .env file
from dotenv import load_dotenv
load_dotenv()

# Add project root to path (in case notebook is run from a subdirectory)
project_root = os.path.dirname(os.path.abspath("__file__"))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Verify environment
provider = os.getenv("LLM_PROVIDER", "anthropic")
max_iters = os.getenv("MAX_ITERATIONS", "6")

print(f"LLM Provider  : {provider}")
print(f"Max Iterations: {max_iters}")
print(f"Tavily Key set: {'YES' if os.getenv('TAVILY_API_KEY') else 'NO (simulated search)'}")

# Import core project modules
from graph import build_graph
from state import OrchestratorState
from langchain_core.messages import HumanMessage

print("\nAll imports successful!")

## Cell 2 — Graph Structure Visualization

In [ ]:
# Build the graph
graph = build_graph()

print("Graph nodes:")
for node in graph.graph.nodes:
    print(f"  - {node}")

print("\nGraph edges:")
for edge in graph.graph.edges:
    print(f"  {edge[0]} --> {edge[1]}")

# Attempt Mermaid visualization (requires IPython display support)
try:
    from IPython.display import Image, display
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception as e:
    print(f"\nMermaid PNG rendering not available ({e})")
    print("\nMermaid diagram (paste into https://mermaid.live):")
    print(graph.get_graph().draw_mermaid())

## Helper: run_demo_query

A convenience function that runs a query and displays results in a structured way.

In [ ]:
def run_demo_query(query: str, verbose: bool = True) -> dict:
    """Run a query through the orchestrator and display results."""
    graph = build_graph()

    initial_state = {
        "messages": [HumanMessage(content=query)],
        "user_query": query,
        "next_agent": "research",
        "research_data": None,
        "analysis_result": None,
        "task_completed": False,
        "iteration_count": 0,
        "agent_contributions": {},
        "validation_notes": None,
    }

    if verbose:
        print(f"\n{'='*70}")
        print(f"QUERY: {query}")
        print(f"{'='*70}")

    # Stream execution
    final_state = initial_state.copy()

    for step in graph.stream(initial_state, stream_mode="updates"):
        for node_name, node_output in step.items():
            if verbose:
                print(f"\n[{node_name.upper()}]")
                if node_name == "supervisor":
                    decision = node_output.get("next_agent", "?")
                    notes = node_output.get("validation_notes", "")
                    print(f"  -> Decision: {decision}")
                    if notes:
                        print(f"  -> Notes: {notes}")
                elif node_name == "research":
                    rd = node_output.get("research_data", "")
                    if rd:
                        preview = rd[:400] + "..." if len(rd) > 400 else rd
                        print(f"  Research preview: {preview}")
                elif node_name == "analyst":
                    ar = node_output.get("analysis_result", "")
                    if ar:
                        preview = ar[:400] + "..." if len(ar) > 400 else ar
                        print(f"  Analysis preview: {preview}")

            # Merge state
            if isinstance(node_output, dict):
                for k, v in node_output.items():
                    if k == "messages":
                        final_state["messages"] = final_state.get("messages", []) + (v or [])
                    elif k == "agent_contributions":
                        existing = final_state.get("agent_contributions", {})
                        final_state["agent_contributions"] = {**existing, **(v or {})}
                    elif v is not None:
                        final_state[k] = v

    if verbose:
        print(f"\n{'='*70}")
        print("FINAL RESULTS")
        print(f"{'='*70}")
        print(f"Completed: {final_state.get('task_completed', False)}")
        print(f"Iterations: {final_state.get('iteration_count', 0)}")
        print(f"Agents used: {list(final_state.get('agent_contributions', {}).keys())}")

        ar = final_state.get("analysis_result", "")
        if ar:
            print(f"\nFinal Analysis Result:")
            print(ar[:800] + ("..." if len(ar) > 800 else ""))

    return final_state


print("Helper function loaded.")

## Cell 3 — Example Query 1: AI Regulation + Sentiment Analysis

**Task**: Research the current state of AI regulation in the EU, then analyze the sentiment of public opinion.

In [ ]:
query1 = (
    "What is the current state of AI regulation in the EU? "
    "Analyze the sentiment of public opinion on this topic."
)

result1 = run_demo_query(query1, verbose=True)

In [ ]:
# Inspect the full research data
print("RESEARCH DATA:")
print(result1.get("research_data", "No research data")[:1000])

In [ ]:
# Inspect the full analysis result
print("ANALYSIS RESULT:")
print(result1.get("analysis_result", "No analysis result")[:1000])

## Cell 4 — Example Query 2: Climate Change Economics + Math

**Task**: Research economic impacts of climate change and calculate projected GDP loss percentages.

In [ ]:
query2 = (
    "What are the economic impacts of climate change? "
    "Calculate the projected GDP loss percentages mentioned in recent studies."
)

result2 = run_demo_query(query2, verbose=True)

In [ ]:
# Show math calculations if any were made
print("Agent Contributions:")
for agent, contrib in result2.get("agent_contributions", {}).items():
    print(f"\n[{agent.upper()}]")
    print(contrib[:800])

## Cell 5 — Example Query 3: JSON Data Validation

**Task**: Validate a Tesla revenue JSON object against a schema.

In [ ]:
query3 = (
    'Validate this JSON data: {"company": "Tesla", "revenue": 97690000000} '
    "against a schema that requires company name (string), revenue as number, and year (number). "
    "What fields are missing or incorrect?"
)

result3 = run_demo_query(query3, verbose=True)

In [ ]:
# Direct tool usage demonstration
print("Direct data_validator tool demo:")
print("-" * 50)

from tools.analysis_tools import data_validator

test_data = '{"company": "Tesla", "revenue": 97690000000}'
schema_desc = "requires company (string), revenue (number), year (number)"

validation_result = data_validator.invoke({
    "data": test_data,
    "schema_description": schema_desc
})

parsed = json.loads(validation_result)
print(f"Valid: {parsed['valid']}")
print(f"Errors: {parsed['errors']}")
print(f"Warnings: {parsed['warnings']}")
print(f"Summary: {parsed['summary']}")

## Bonus: Direct Tool Testing

In [ ]:
from tools.analysis_tools import sentiment_analyzer, math_calculator, text_summarizer
from tools.search_tools import web_search, wikipedia_search

# Sentiment analysis
print("=== Sentiment Analyzer ===")
texts = [
    "I absolutely love this incredible technology!",
    "This is a terrible failure and completely useless.",
    "The report was submitted on Tuesday.",
]
for t in texts:
    result = json.loads(sentiment_analyzer.invoke(t))
    print(f"  '{t[:50]}...' => {result['label']} (polarity={result['polarity']:.3f})")

print()

# Math calculator
print("=== Math Calculator ===")
expressions = ["2 + 2", "sqrt(144)", "2**10", "log(e)", "factorial(5)"]
for expr in expressions:
    result = math_calculator.invoke(expr)
    print(f"  {expr} = {result}")

In [ ]:
# Text summarizer
print("=== Text Summarizer ===")
long_text = """
Artificial intelligence (AI) is rapidly transforming every sector of the global economy.
From healthcare to finance, transportation to education, AI-powered systems are automating
complex tasks, improving decision-making, and creating new opportunities for innovation.
Machine learning models trained on vast datasets can now diagnose diseases with accuracy
comparable to specialist physicians. Autonomous vehicles are reducing traffic fatalities.
Natural language processing enables real-time translation across hundreds of languages.
However, these advances also raise important questions about job displacement, algorithmic
bias, data privacy, and the concentration of power in a handful of technology companies.
Policymakers around the world are grappling with how to regulate AI effectively without
stifling innovation. The EU AI Act represents the most comprehensive regulatory framework
to date, establishing risk tiers and requirements for transparency and accountability.
"""

summary = text_summarizer.invoke({"text": long_text.strip(), "max_words": 50})
print(f"Original word count: {len(long_text.split())}")
print(f"Summary word count: {len(summary.split())}")
print(f"Summary:\n{summary}")

In [ ]:
# Wikipedia search
print("=== Wikipedia Search ===")
wiki_result = wikipedia_search.invoke("Large language model")
print(wiki_result[:500])